In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% ! important;}
div.cell.code_cell.rendered{width:100%}
div.input_prompt{padding:0px}
div.CodeMirror {font-family:Consolas ; font-size:12pt;}
div.text_cell_render.rendered_html {font-size:12pt;}
div.output {font-size:12pt; font-weight:bold}
div.input {font-family:Consolas ; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper {padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

# <span style="color:rgb(0,200,100);">1. 패키지 </span>

In [ ]:
# python-dotenv, langchain(1.2.0), langchain-openai, langchain-pinecone
# pandas,  langchain-community, docx2txt,  langchain-text_splitters,  langchain-ollama

# <span style="color:rgb(0,200,100);">2. 환경설정(환경변수, 시스템파라미터변수) </span>

In [96]:
from dotenv import load_dotenv
import os
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

OPENAI_LLM_MODEL = 'gpt-4o-mini'
OPENAI_EMBEDDING_MODEL = "text-embedding-3-large" # 차원수 3072

PINECONE_INDEX_NAME = "better-rag-index"
PINECONE_INDEX_DEMENSION=3072
PINECONE_INDEX_METRIC = "cosine"
PINECONE_INDEX_REGION="us-east-1"
PINECONE_INDEX_CLOUD="aws"

# <span style="color:rgb(0,200,100);">3. 문서를 chunk로 분할하기 </span>

In [11]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./data/소득세법_with_markdown.docx')
document = loader.load()
text_spliter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", '\n', " ", ""]
)
# documents = loader.load_and_split(text_spliter)

documents = text_spliter.split_documents(document)
print(f'총 {len(documents)} 캐 청크 크기')

총 194 캐 청크 크기


# <span style="color:rgb(0,200,100);">4. metadata 추가하기 </span>
- 청크 내용의 카테고리, 청크 내용의 title, 조항

In [28]:
import re
def remove_special_chars(text:str)-> str:
    "특수문자 밑 \n 제거(:은 그대로)"
    # \n \제거
    text = text.replace("\n", " ")
    # 한글, 영문, 숫자, 공백, 마침표, 콤마, :만 남기고 전부 제거
    cleaned = re.sub("[^가-힣a-zA-Z0-9\s,\.:]",'',text)
    # 불필요한 중복 공백을 제거
    cleaned = re.sub(r"\s+",' ', cleaned).strip()
    return cleaned
# 사용예시
content ="**소득세 납세 의무** :\n\n이 법 또는 「법인세법」에 따라 소득에 대한 소득세를 부과"
print(remove_special_chars(content))

소득세 납세 의무 : 이 법 또는 법인세법에 따라 소득에 대한 소득세를 부과


In [30]:
# 제목을 추출하는 함수(exaone3.5) :cmd창에서 ollama run exaone3.5:2.4b
from langchain_ollama import ChatOllama
def extract_title_with_llm(content):
    "exaone3.5를 사용하여 content의 제목을 추출"
    title_extractor_llm = ChatOllama(
        model="exaone3.5:2.4b",
        temperature =0.1,
        num_predict = 30 # 최대 30토큰까지 출력
    )
    prompt =f"""다음 소득세법 조문의 핵심 제목을 30토큰 이내로 간단히 완벽하게 말이 되도록 추출해 주세요.
    중간에 말이 끊기면 안 되요.
    예 : 소득세 납세의미 범위 : 공동사업자별 소득과세, 상속과제, 증여자
    조문 : {content}"""
    
    ai_message = title_extractor_llm.invoke(prompt)
    title = ai_message.content.strip()
    return remove_special_chars(title)
content=""" ① 소득세의 과세기간은 1월 1일부터 12월 31일까지 1년으로 한다.
② 거주자가 사망한 경우의 과세기간은 1월 1일부터 사망한 날까지로 한다.
③ 거주자가 주소 또는 거소를 국외로 이전(이하 “출국”이라 한다)하여 비거주자가 되는 경우의 과세기간은 1월 1일부터 출국한 날까지로 한다.
"""
print(extract_title_with_llm(content))

소득세 과세기간 정의: 연간 1월 1일부터 12월 31일까지 과세,


In [36]:
def categorize_content(content):
    "내용 카테고리 분류"
    # 쳇 gpt가 meta 데이터에 쓴다고 하니 만들어 준 카테고리
    categories = {
    '납세의무': ['납세의무', '거주자', '비거주자', '원천징수', '공동사업자', '상속인', '증여자', '신탁재산'],
    '세율계산': ['세율', '과세표준', '산출세액', '결정세액', '종합소득', '퇴직소득'],
    '근로소득': ['근로소득', '총급여', '급여', '연봉', '임금', '퇴직소득'],
    '사업소득': ['사업소득', '공동사업', '주택임대소득', '부업소득'],
    '이자배당': ['이자소득', '배당소득', '예금이자', '채권', '의제배당'],
    '양도소득': ['양도소득', '자산양도', '부동산양도', '주식양도'],
    '연금소득': ['연금소득', '공적연금', '사적연금', '연금보험'],
    '기타소득': ['기타소득', '상금', '보상금', '발명보상금', '종교인소득'],
    '공제감면': ['공제', '소득공제', '세액공제', '기본공제', '감면'],
    '비과세': ['비과세', '면제', '복무급여', '실업급여', '출산휴가급여', '장학금'],
    '신고납부': ['신고', '납부', '납세지', '신고기한', '원천징수'],
    '과세기간': ['과세기간', '과세연도', '사업연도'],
    '과세소득구분': ['종합소득', '퇴직소득', '양도소득', '금융투자소득'],
    }
    result = [] # 각 카테고리 추가
    for category, keywords in categories.items():
        #print(keywords)
        #print(any([keyword in  content for keyword in  keywords]))
        if any([keyword in  content for keyword in  keywords]):
            result.append(category)
    if not result:
        result.append("기타")
    return result
categorize_content(content)

['납세의무', '과세기간']

- 개선된 카테고리 분류 방법 : 키워드의 중요도에 따라 가중치를 부여하고, 가중치가 높은 순으로 카테고리 반환

In [37]:
def get_category():
    return {
        '납세의무': {
            '납세의무': 3, '거주자': 3, '비거주자': 3, '납세의무자': 3, 
            '원천징수': 2, '원천징수의무자': 2, '공동사업자': 2, 
            '상속인': 2, '증여자': 2, '신탁재산': 2
        },
        '세율계산': {
            '세율': 3, '소득세': 3, '과세표준': 3, '산출세액': 3, '세액': 3,
            '결정세액': 2, '세액계산': 2, '기본세율': 2, '세율적용': 2,
            '누진세율': 2, '종합소득세': 2
        },
        '근로소득': {
            '근로소득': 3, '총급여': 3, '급여': 3, '연봉': 3, '임금': 3,
            '근로소득금액': 2, '총급여액': 2, '상여': 2, '수당': 2,
            '봉급': 2, '직장인': 2
        },
        '사업소득': {
            '사업소득': 3, '총수입금액': 3, '필요경비': 3,
            '사업자': 2, '사업소득금액': 2, '결손금': 2, '이월결손금': 2,
            '주택임대소득': 2, '공동사업': 2
        },
        '이자배당': {
            '이자소득': 3, '배당소득': 3, '예금이자': 2, '채권': 2,
            '의제배당': 2, '배당세액공제': 2, '분리과세이자소득': 2,
            '분리과세배당소득': 2
        },
        '양도소득': {
            '양도소득': 3, '자산양도': 2, '부동산양도': 2, '주식양도': 2,
            '양도차익': 2, '취득가액': 2, '양도가액': 2, '양도소득금액': 2
        },
        '연금소득': {
            '연금소득': 3, '연금계좌': 3, '연금저축': 3,
            '퇴직연금': 2, '공적연금': 2, '사적연금': 2, '연금보험': 2,
            '연금수령': 2
        },
        '기타소득': {
            '기타소득': 3, '가상자산': 3, '가상자산소득': 2,
            '상금': 2, '보상금': 2, '종교인소득': 2, '원고료': 2,
            '복권': 1, '당첨금': 1, '발명보상금': 1
        },
        '공제감면': {
            '공제': 3, '소득공제': 3, '세액공제': 3,
            '기본공제': 2, '인적공제': 2, '특별공제': 2, '추가공제': 2,
            '근로소득공제': 2, '연금소득공제': 2, '퇴직소득공제': 2,
            '연금계좌세액공제': 2, '감면': 2
        },
        '비과세': {
            '비과세': 3, '비과세소득': 3, '면제': 2, '세액감면': 2,
            '소득세면제': 2, '복무급여': 1, '실업급여': 1, 
            '출산휴가급여': 1, '장학금': 1
        },
        '신고납부': {
            '신고': 3, '확정신고': 3, '과세표준확정신고': 3, 
            '납부': 3, '중간예납': 2, '가산세': 2,
            '신고기한': 2, '납부기한': 2, '납세지': 2
        },
        '과세기간': {
            '과세기간': 3, '과세연도': 3, '사업연도': 2, 
            '과세기간종료일': 2
        },
        '과세방식': {
            '종합과세': 3, '분리과세': 3, '합산과세': 2, 
            '종합소득과세표준': 2, '분리과세소득': 2, 
            '금융투자소득': 2
        },
        '장부기장': {
            '장부': 3, '복식부기': 3, '간편장부': 2, '기장': 2,
            '장부기록': 2, '증명서류': 2, '기장세액공제': 2
        }
    }

In [53]:
def categorize_content(content, top_k=None):
    '''내용 카테고리 분류 -점수 기반으로 모든 카테고리를 점수 순으로 반환
     parameters :
     - content : 분류할 텍트스 내용
     - top_k : 상위 몇 개 까지 카테고리를 반환할지(None이면 모든 카테고리 반환)
     Returns:
     - 카테고리 리스트(점수 높은 순)'''
    category_keywords = get_category()
    category_scores = {}
    # 각 카테고리별 점수 계산
    for category, weigthed_keywords in category_keywords.items():
        #print(category, weigthed_keywords)
        score=0
        for keyword, weight in weigthed_keywords.items():
            if keyword in content:
                score += weight
        if score > 0:
            category_scores[category] = score
    #print(category_scores)
    
    # 내림차순 정렬한 카테고리 이름만 추출
    sorted_categories = sorted(category_scores.items(), key=lambda x:x[1], reverse=True)
    all_categoris = [category[0] for category in sorted_categories]
    #print(all_categoris)
    # 매칭되는 카테고리가 없으면 "기타" 반환
    if not all_categoris:
        all_categoris = ["기타"]
    # top_k가 지정되면 상위 top_k개만, 아니면 전체 반환
    if top_k is not None:
        return all_categoris[:top_k]
    else:
        return all_categoris
categorize_content(content, 2)

['납세의무', '세율계산']

In [54]:
categorize_content("연봉 5천만원인 회사원의 소득세는 얼마에요?")

['세율계산', '근로소득']

In [57]:
categorize_content(documents[46].page_content)

['세율계산', '이자배당', '공제감면', '과세방식', '납세의무', '사업소득', '과세기간']

In [84]:
# 헤당 조항 추출하기
import re
def get_article(content):
    "조항들 추출"
    article = re.findall(r"제(\d+)조", content)
    article = list(set(article))
    article.sort(key=lambda x:int(x))
    # [f"{a}조" for a in article]
    if article:
        return "「"+ ",".join([f"{a}조" for a in article]) +"」"
print(get_aritcle("무시기"))

None


In [97]:
%%time
# 메타데이터(source,title, category, article(조항))를 포함한 새로운 chunk
enhanced_chunks = []
for i, chunk in enumerate(documents):
    if i%20 == 0:
        print(f"진행중 : {i/len(documents)*100:.2f}% 진행")
    content =chunk.page_content
    metadata = chunk.metadata.copy()
    
    #metadata["title"] = extract_title_with_llm(content) # exaone으로 제목 추출
    metadata["category"] = categorize_content(content) # 카테고리 추출
    metadata["chunk_id"] = f"chunk_{i:03d}"
    article = get_article(content)
    if article:
        metadata["article"] = article
    enhanced_chunks.append(type(chunk)(page_content=content, metadata=metadata))
print("확장된 chunk 처리 완료")

진행중 : 0.00% 진행
진행중 : 10.31% 진행
진행중 : 20.62% 진행
진행중 : 30.93% 진행
진행중 : 41.24% 진행
진행중 : 51.55% 진행
진행중 : 61.86% 진행
진행중 : 72.16% 진행
진행중 : 82.47% 진행
진행중 : 92.78% 진행
확장된 chunk 처리 완료
CPU times: total: 46.9 ms
Wall time: 48 ms


# <span style="color:rgb(0,200,100);">5. 임베딩 모델 설정 </span>

In [99]:
from langchain_openai import OpenAIEmbeddings
embedding = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL
)

# <span style="color:rgb(0,200,100);">6. Pinecone 인덱스 생성 및 vector store(DB) 저장</span>

In [91]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
# pinecone 클라이언트
pc = Pinecone(api_key=PINECONE_API_KEY)
#print("pinecone index들 :", pc.list_indexes().names())

# 인덱스 생성 여부 확인 및 생성
# if PINECONE_INDEX_NAME not in pc.list_indexes().names():
if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=PINECONE_INDEX_DEMENSION,
        metric=PINECONE_INDEX_METRIC,
        spec=ServerlessSpec(region=PINECONE_INDEX_REGION, cloud=PINECONE_INDEX_CLOUD)
    )
    print("인덱스 생성완료")
else:
    print(f"인덱스 {PINECONE_INDEX_NAME}이 이미 존재합니다.")

인덱스 생성완료


In [100]:
%%time
# Pinecone 벡터 스토어 업로드
vector_database = PineconeVectorStore.from_documents(
    documents=enhanced_chunks,
    embedding=embedding,
    index_name = PINECONE_INDEX_NAME,
)
print("벡터 DB 저장 완료")

벡터 DB 저장 완료
CPU times: total: 3.67 s
Wall time: 13.1 s


# <span style="color:rgb(0,200,100);">7. 유사도 검색(meta데이터 활용)</span>

In [101]:
query = "연봉 5천만원인 직장인의 소득세는 얼마예요?"
categorize_content(query)

['근로소득', '세율계산']

In [105]:
# Retriever 생성
retriever = vector_database.as_retriever(
    search_kwargs={
        "k":3,
        "fillter":{"category":{"category":{"$in":categorize_content(query)}}}
    }
)
docs = retriever.invoke(query)
print("관련 문서")
for i,doc in enumerate(docs):
    category = doc.metadata["category"]
    article = doc.metadata.get("article", "조항없음")
    content = doc.page_content[:20]
    print(f"{i+1}.{article} {category} - {content}")

관련 문서
1.「6조,9조,10조,11조,12조,99조」 ['납세의무', '사업소득', '세율계산', '근로소득', '비과세', '신고납부', '과세기간'] - [전문개정 2009. 12. 31.]
2.「20조,21조,22조,47조,59조,146조」 ['근로소득', '사업소득', '연금소득', '공제감면', '비과세', '기타소득', '세율계산', '과세기간', '납세의무'] - 21. 제1호부터 제20호까지의 규정
3.「20조,21조,25조,32조,164조」 ['기타소득', '연금소득', '근로소득'] - 2) 대학의 교직원 또는 대학과 고용


In [ ]:
from langchain_upstage import ChatUpstage, UpstageEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_pinecone import PineconeVectorStore
from dotenv import load_dotenv